# 08 - Kaggle EnMAP L2A Dataset Downloader & Packager

**Arab Youth Space Hackathon 2026** | Team: ليوا الخضراء | Theme: Precision Agriculture & Crop Intelligence

This notebook runs in **Kaggle Notebooks** to download DLR EnMAP Level-2A hyperspectral scenes directly into `/kaggle/working/enmap_liwa_dataset` and generate Kaggle dataset metadata so it can be saved as a reusable Kaggle Dataset.

In [ ]:
# Step 1: Set up output directory following Kaggle pathing invariant
import os
import json
import requests
from tqdm import tqdm

# Output path inside Kaggle working directory
DATASET_DIR = "/kaggle/working/enmap_liwa_dataset"
os.makedirs(DATASET_DIR, exist_ok=True)

print(f"Target dataset output directory: {DATASET_DIR}")


In [ ]:
# Step 2: Configure DLR Ticket / Credentials and Target Scene URLs
# Insert your active DLR auth ticket parameter (e.g., ticket=ST-1800-VC7MSVEe...)
DLR_TICKET = "ST-1800-VC7MSVEe-saNtHuwRokAy7--vd8-auth"

# List of target EnMAP scene URLs for Liwa Oasis
# Example scene files structure from DLR L2A server:
SCENE_TARGETS = [
    {
        "name": "ENMAP_L2A_Liwa_Scene_01",
        "tif_url": f"https://download.geoservice.dlr.de/ENMAP/files/L2A/2024/04/27/DT0000000001/01/ENMAP01-____L2A-DT0000000001_20240427T092849Z_001_V010502_20250926T080128Z/ENMAP01-____L2A-DT0000000001_20240427T092849Z_001_V010502_20250926T080128Z-SPECTRAL_IMAGE.TIF?ticket={DLR_TICKET}",
        "xml_url": f"https://download.geoservice.dlr.de/ENMAP/files/L2A/2024/04/27/DT0000000001/01/ENMAP01-____L2A-DT0000000001_20240427T092849Z_001_V010502_20250926T080128Z/ENMAP01-____L2A-DT0000000001_20240427T092849Z_001_V010502_20250926T080128Z-METADATA.xml?ticket={DLR_TICKET}"
    }
]

print(f"Configured {len(SCENE_TARGETS)} scene targets for download")


In [ ]:
# Step 3: Stream download scene files directly into Kaggle dataset directory
def download_file(url, save_path):
    print(f"Downloading {os.path.basename(save_path)}...")
    response = requests.get(url, stream=True)
    if response.status_code == 200:
        total_size = int(response.headers.get('content-length', 0))
        with open(save_path, 'wb') as f, tqdm(total=total_size, unit='B', unit_scale=True) as bar:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
                    bar.update(len(chunk))
        print(f"Saved: {save_path}")
    else:
        print(f"Failed download (HTTP {response.status_code}): {url}")

for scene in SCENE_TARGETS:
    scene_dir = os.path.join(DATASET_DIR, scene["name"])
    os.makedirs(scene_dir, exist_ok=True)
    
    tif_path = os.path.join(scene_dir, "SPECTRAL_IMAGE.TIF")
    xml_path = os.path.join(scene_dir, "METADATA.xml")
    
    if not os.path.exists(tif_path):
        download_file(scene["tif_url"], tif_path)
    if not os.path.exists(xml_path):
        download_file(scene["xml_url"], xml_path)


In [ ]:
# Step 4: Generate dataset-metadata.json for Kaggle Dataset creation
metadata = {
    "title": "EnMAP Hyperspectral Liwa Oasis L2A Dataset",
    "id": "assemelqersh/enmap-liwa-l2a",
    "licenses": [{"name": "CC-BY-4.0"}]
}

metadata_path = os.path.join(DATASET_DIR, "dataset-metadata.json")
with open(metadata_path, "w") as f:
    json.dump(metadata, f, indent=2)

print(f"Kaggle dataset metadata written to {metadata_path}")


In [ ]:
# Step 5: Verify downloaded files in dataset directory
for root, dirs, files in os.walk(DATASET_DIR):
    for file in files:
        full_p = os.path.join(root, file)
        size_mb = os.path.getsize(full_p) / (1024 * 1024)
        print(f"{full_p} ({size_mb:.2f} MB)")
